# 13.2 프롬프팅 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter13_2_prompting_cot.ipynb)

책 본문: [13.2 프롬프팅](https://smhanlab.com/book-ml/kor/ml1/chapter13/2.html)

이 노트북은 책 13.2절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. 13.1절 연습문제 1의 `next_token_distribution` 구현 — zero-shot vs few-shot 대조(본문 코드 블록 재현)
2. `temperature`를 붙인 장난감 모델 — 같은 "사과는" 프롬프트에서 \(T=1.0\) vs \(T=3.0\) 샘플링 비교
3. temperature 곡선 — logits \([2.0, 1.0, 0.5, 0.2]\)에 \(T \in [0.2, 3.0]\)를 적용할 때 최상위 확률의 변화(본문 `ch13_2_temperature.svg` 생성)
4. CoT vs 직접 답 확률 시뮬레이션 — 3단계 추론(각 부분 추론: 조건 0.9 vs 조건없음 0.5), 4,000회 반복: 직접 ≈ \(0.5^3 = 0.125\), CoT ≈ \(0.9^3 = 0.729\)

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. `next_token_distribution`: zero-shot vs few-shot

13.1절 연습문제 1의 함수를 완성한 것입니다 — `context_word`가 등장한 모든 위치에서
바로 다음 토큰을 세어 확률로 바꿉니다. 본문의 zero-shot / few-shot 예시를 그대로
재현합니다: **프롬프트 안의 토큰이 곧 다음 토큰 예측의 '조건'이 된다**는 기계적
원리가 이 빈도 세기 모델에서도 그대로 성립해야 합니다.

In [2]:
def next_token_distribution(corpus_tokens, context_word):
    counts = {}
    for i in range(len(corpus_tokens) - 1):
        if corpus_tokens[i] == context_word:
            nxt = corpus_tokens[i + 1]
            counts[nxt] = counts.get(nxt, 0) + 1
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}

# 13.1절 연습문제 검증
assert next_token_distribution(
    "나는 학교에 간다 나는 집에 간다 나는 학교에 있다".split(), "나는") == \
    {"학교에": 2/3, "집에": 1/3}

# zero-shot: "사과는"가 문맥에 등장한 적이 없어 예측 불가 -> {}
zero_shot_context = "오늘 기분이 좋다 사과는".split()
print("zero-shot '사과는' ->", next_token_distribution(zero_shot_context, "사과는"))

# few-shot: "OO는 OO이다" 예시 2개 포함 -> {'과일이다': 1.0}
few_shot_context = "사과는 과일이다 바나나는 과일이다 자동차는 기계이다 사과는".split()
print("few-shot  '사과는' ->", next_token_distribution(few_shot_context, "사과는"))

# 장난감 모델의 한계: 예시에 없던 새 단어 '비행기는'에는 일반화하지 못한다 -> {}
new_word = "비행키는"
print("새 단어   '" + new_word + "' ->", next_token_distribution(few_shot_context, new_word))
print("-> 본문과 일치: few-shot 예시는 '가중치'가 아니라 '조건부 확률의 조건'이다.")

zero-shot '사과는' -> {}
few-shot  '사과는' -> {'과일이다': 1.0}
새 단어   '비행키는' -> {}
-> 본문과 일치: few-shot 예시는 '가중치'가 아니라 '조건부 확률의 조건'이다.


## 2. temperature: 같은 프롬프트에서 왜 매번 다른 답이 나오나

13.1절에서 다음 토큰 예측이 **확률분포**를 반환한다고 배웠다. 그 분포에서
토큰을 고르는 방법을 바꾸는 것이 **temperature \(T\)**다 — logits를 \(T\)로
나눈 뒤 softmax:

\[P(i) = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}\]

\(T<1\)이면 분포가 가팔라지고(집중), \(T>1\)이면 평탄해진다(다양).
장난감 모델에 fixed vocabulary \(\{\text{과일이다},\ \text{나무이다}\}\)와
작은 균일 prior(0.1)를 붙여, **같은 "사과는" 프롬프트**에서 \(T=1.0\)과
\(T=3.0\)으로 20회 샘플링합니다 — 두 후보가 *확률적으로* 나올 수 있음을
확인합니다.

In [3]:
def softmax(x):
    x = np.asarray(x, float)
    e = np.exp(x - x.max())
    return e / e.sum()

def toy_next_token(context_words, last_word, T=1.0, rng=None):
    # fixed vocab + 빈도 + 균일 prior 0.1 -> softmax(logits / T)
    V = ["과일이다", "나무이다"]
    counts = {w: 0 for w in V}
    for i in range(len(context_words) - 1):
        if context_words[i] == last_word and context_words[i + 1] in counts:
            counts[context_words[i + 1]] += 1
    logits = np.array([counts[w] for w in V], float) + 0.1
    p = softmax(logits / T)
    if rng is None:
        return V[int(np.argmax(p))], p
    return V[int(rng.choice(len(V), p=p))], p

from collections import Counter
rng = np.random.default_rng(7)
s1 = [toy_next_token(few_shot_context, "사과는", T=1.0, rng=rng)[0] for _ in range(20)]
print("T=1.0  20회 샘플:", s1)
print("   ->", dict(Counter(s1)))

rng = np.random.default_rng(7)
s3 = [toy_next_token(few_shot_context, "사과는", T=3.0, rng=rng)[0] for _ in range(20)]
print("T=3.0  20회 샘플:", s3)
print("   ->", dict(Counter(s3)))
print("-> T가 클수록 '과일이다' 비율이 낮아진다(prior가 더 크게 반영).")
print("   same prompt, different outputs = T>0 sampling의 본질.")

T=1.0  20회 샘플: ['과일이다', '나무이다', '나무이다', '과일이다', '과일이다', '나무이다', '과일이다', '나무이다', '나무이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '나무이다', '나무이다', '과일이다', '나무이다']
   -> {'과일이다': 12, '나무이다': 8}
T=3.0  20회 샘플: ['나무이다', '나무이다', '나무이다', '과일이다', '과일이다', '나무이다', '과일이다', '나무이다', '나무이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '과일이다', '나무이다', '나무이다', '나무이다', '나무이다']
   -> {'나무이다': 10, '과일이다': 10}
-> T가 클수록 '과일이다' 비율이 낮아진다(prior가 더 크게 반영).
   same prompt, different outputs = T>0 sampling의 본질.


## 3. temperature 곡선 — 본문의 표 재현 + `ch13_2_temperature.svg`

logits \([2.0, 1.0, 0.5, 0.2]\)에 \(T \in [0.2, 3.0]\)을 적용할 때
최상위 토큰의 확률(왼쪽)과, \(T=0.5/1.0/2.0\)에서의 4차원 분포(오른쪽)를
그립니다. 본문의 표(0.5 → 0.825, 1.0 → 0.569, 2.0 → 0.402)와 일치해야 합니다.

In [4]:
logits = np.array([2.0, 1.0, 0.5, 0.2])
Ts = np.linspace(0.2, 3.0, 57)
top_probs = [softmax(logits / T)[0] for T in Ts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.8))

# 왼쪽: T -> 최상위 확률
ax1.plot(Ts, top_probs, lw=2, color="#1f77b4")
ax1.axhline(1.0, color="gray", lw=0.8, ls="--", alpha=0.6)
ax1.set_xlabel("temperature T")
ax1.set_ylabel("P(top token)")
ax1.set_title("Lower T sharpens the distribution\n(T→0: one-hot, T→∞: uniform)", fontsize=11)
ax1.set_ylim(0, 1.05)
ax1.set_xlim(0.2, 3.0)
ax1.grid(alpha=0.3)

# 오른쪽: T=0.5 / 1.0 / 2.0 분포 막대
bar_Ts = [0.5, 1.0, 2.0]
x = np.arange(4)
width = 0.26
for i, T in enumerate(bar_Ts):
    p = softmax(logits / T)
    ax2.bar(x + (i - 1) * width, p, width=width, label=f"T={T}")
ax2.set_xticks(x + width)
ax2.set_xticklabels(["t1", "t2", "t3", "t4"])
ax2.set_ylabel("Probability")
ax2.set_title("Same logits [2.0, 1.0, 0.5, 0.2]\nHigher T flattens the distribution", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis="y")

fig.suptitle("temperature — a parameter controlling the 'width of choices'", y=1.03)
fig.tight_layout()
fig.savefig(IMG + "/ch13_2_temperature.svg", bbox_inches="tight")
plt.show()
print("figure saved -> kor/src/images/ch13_2_temperature.svg")

# 본문의 표 수치 검증
for T, expect_top in [(0.5, 0.825), (1.0, 0.569), (2.0, 0.402)]:
    top = softmax(logits / T)[0]
    assert abs(top - expect_top) < 0.002, (T, top, expect_top)
    print(f"T={T}: top={top:.3f} (본문 표 {expect_top}) OK")
print("-> 본문 표와 정확히 일치.")

figure saved -> kor/src/images/ch13_2_temperature.svg
T=0.5: top=0.825 (본문 표 0.825) OK
T=1.0: top=0.569 (본문 표 0.569) OK
T=2.0: top=0.402 (본문 표 0.402) OK
-> 본문 표와 정확히 일치.


## 4. CoT vs 직접 답 — '중간 결과를 다음 단계의 조건으로 쓰느냐'

본문의 수치 모델: 문제가 \(d\)단계 부분 추론으로 분해된다. 각 부분 추론은
*중간 결과를 조건으로 쓸 때* 성공률 0.9, *조건 없이 한 방에* 맞출 때 0.5.

- **직접(한 방에)**: 조건 없음 → 각 부분 추론 0.5 → 전체 \(0.5^3 = 0.125\)
- **CoT(단계별로)**: 중간 결과를 조건으로 → 각 부분 추론 0.9 → 전체 \(0.9^3 = 0.729\)

핵심 메커니즘 — *모델이 쓴 중간 결과가 다음 단계의 조건이 된다* — 가
"부분 추론 하나당 0.5 → 0.9"로 들어가는 것. 4,000회 반복 시뮬레이션으로
확인합니다(각 실행은 부분 추론 3개를 독립적으로 시도, 한 번만 틀어져도
최종 오답).

In [5]:
rng = np.random.default_rng(12)
N = 4000
p_cond = 0.9   # 부분 추론 하나: 중간 결과를 '조건'으로 쓸 때의 성공률
p_noc = 0.5    # 부분 추론 하나: 조건 없이 한 방에 맞출 때의 성공률

# direct: 조건 없음 -> 각 부분 추론 0.5, 3개 모두 맞아야 최종 정답
direct_ok = int((rng.random((N, 3)) < p_noc).all(axis=1).sum())

# CoT: 각 부분 추론이 '이전 결과를 조건으로' 0.9, 3개 모두 맞아야 정답
cot_ok = int((rng.random((N, 3)) < p_cond).all(axis=1).sum())

print(f"직접: {direct_ok}/{N} = {direct_ok/N:.3f}  (이론 0.5^3 = {0.5**3:.3f})")
print(f"CoT : {cot_ok}/{N} = {cot_ok/N:.3f}  (이론 0.9^3 = {0.9**3:.3f})")
assert abs(direct_ok/N - 0.5**3) < 0.02
assert abs(cot_ok/N - 0.9**3) < 0.02
print("-> 본문의 0.125 vs 0.729 시나리오 재현.")

# 단계 수를 늘리면 격차가 얼마나 벌어지는지 (본문: '단계가 많아질수록')
print("\n단계 수 (조건 0.9 vs 조건없음 0.5, d단계 전체 성공률):")
for d in [2, 3, 5, 10]:
    cot_p = 0.9 ** d
    direct_p = 0.5 ** d
    print(f"  {d}단계: CoT={cot_p:.3f}  direct={direct_p:.4f}  격차={cot_p-direct_p:.3f}")

직접: 493/4000 = 0.123  (이론 0.5^3 = 0.125)
CoT : 2945/4000 = 0.736  (이론 0.9^3 = 0.729)
-> 본문의 0.125 vs 0.729 시나리오 재현.

단계 수 (조건 0.9 vs 조건없음 0.5, d단계 전체 성공률):
  2단계: CoT=0.810  direct=0.2500  격차=0.560
  3단계: CoT=0.729  direct=0.1250  격차=0.604
  5단계: CoT=0.590  direct=0.0312  격차=0.559
  10단계: CoT=0.349  direct=0.0010  격차=0.348


## 5. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| `next_token_distribution` zero vs few | `{}` vs `{'과일이다': 1.0}` | 프롬프트 토큰 = '조건부 확률의 조건' (가중치가 아님) |
| fixed-vocab + prior, T=1.0 vs 3.0 | '과일이다' 비율이 T에 따라 변동 | next-token = 확률분포 → sampling이 "매번 다른 답"을 만든다 |
| temperature 곡선 | top 확률 0.825/0.569/0.402 (T=0.5/1/2) | T가 '선택지 폭'을 조절: T↓ 집중, T↑ 다양 |
| CoT vs direct 시뮬레이션 (4,000회) | 0.125 vs 0.73 (3단계) | 중간 결과를 *다음 단계의 조건*으로 쓰면 부분 추론 성공률 0.5→0.9 |

**다음 13.3절**: 사전학습+SFT+프롬프팅의 '세 겹' 중 *모델 쪽*을 바꾸는
방법 — SFT, RLHF(보상모델 + PPO), DPO. 이번 절의 "프롬프트로 가리키는
능력"을 *가중치에 새기는* 단계입니다.